# Constant Folding

Constant folding is a compile-time optimization where expressions composed entirely of constants are evaluated in advance, and the result replaces the original expression in the program or computational graph.

Formally:

If an expression

$$
E = f(c_1,c_2,...,c_n)
$$

where all $c_i$ are constants and $f$ is deterministic, then the compiler replaces $E$ with its computed value.

## Simple Programming Example

### Before optimization

In [ ]:
int x = 3 * 4;
int y = x + 2;

The compiler transforms:

3 * 4  →  12

### After constant folding

In [ ]:
int x = 12;
int y = x + 2;

No runtime multiplication is needed.

## In Deep Learning / Computational Graphs

Consider a graph:

In [ ]:
A = constant tensor
B = constant tensor
C = A + B
D = Conv(X, W)
E = D + C

Since A + B depends only on constants:

In [ ]:
C = precomputed_tensor

The graph becomes:

In [ ]:
D = Conv(X, W)
E = D + precomputed_tensor

The add between A and B disappears at inference time.

## Why It Matters

1. Reduces runtime computation

No need to recompute fixed expressions every inference step.

2. Shrinks graph size

Fewer nodes → fewer kernel launches → better scheduling.

3. Enables further optimizations

Constant folding often exposes new optimization opportunities:
- Dead code elimination
- Operator fusion
- Weight precomputation

## Example in ML Context

### BatchNorm folding into Conv

During inference:

$$
y = \gamma . \frac{Wx + b - \mu}{\sigma} + \beta
$$

Since $\gamma, \mu, \sigma, \beta$ are constants after training, we precompute new weights:

$$
W' = W.\frac{\gamma}{\sigma}
$$

$$
b' = \gamma . \frac{b - \mu}{\sigma} + \beta
$$

Now BatchNorm disappears.

This is a form of constant folding applied to model parameters.

## Important Distinction

Constant folding is compile-time evaluation, not:
- Operator fusion (merging ops)
- Quantization
- Pruning

It specifically means:
- Replace constant expressions with their computed value.

## Where You See It

- C/C++ compilers (LLVM, GCC)
- Python JIT (PyTorch TorchScript)
- TensorFlow Graph Optimization
- ONNX optimization passes
- TVM / XLA
- Mobile inference engines (TFLite, CoreML)

## When It Cannot Apply

Constant folding cannot be applied if:
- Expression depends on runtime input
- Operation is non-deterministic
- It has side effects